<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.3-agent-mcp/notebooks/GCP_Capstone_7.3_AgentMCP.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.3 Connect an Agent to the Lane — McpToolset, a Credential That Refreshes, Three Identities in One Request
**Netsetos GenAI Engineering — GCP Capstone** · Module 7 · rebuilt on the live lane, 8 September 2026

An ADK agent over `documind-mcp`, the service 7.2 deployed. Every cell runs against the lane: the agent discovers the four tools at runtime, presents a fresh credential on every request, names its tenant, and answers with the lane's citations. Then one agent identity across three tenants, the outsider seen from the agent's seat, the audit line read back, and the proof that the agent and the API cite the same chunk.


## Setup


In [ ]:
!pip install -q "google-adk[mcp]==2.8.0" fastmcp==3.4.7 google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

# BOTH PINS ARE LOAD-BEARING, and the failure they prevent is quieter than a version conflict.
# `pip install google-adk fastmcp` resolves: base google-adk declares mcp>=1.24,<2 only under its
# EXTRAS, so nothing constrains the plain install - you get adk 2.8.0 + fastmcp 4 + mcp 2, pip
# reports success, and `from google.adk.tools.mcp_tool import McpToolset` fails at RUNTIME.
# `google-adk[mcp]` asks for the extra explicitly, at which point pip can SEE the conflict, and
# fastmcp 3.4.7 - the server's pin (7.1) - is the resolution. Re-check when you bump either.

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (7.2 deployed documind-mcp there)
REGION     = "us-central1"
TENANT     = "acme"

import os, subprocess, time
from datetime import datetime, timedelta, timezone
import requests
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import AuthorizedSession, Request

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"      # Gemini 3.x generation is served from the global endpoint
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

creds, _ = google.auth.default()
NUMBER  = AuthorizedSession(creds).get(f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
MCP_URL = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # 7.2's service, deterministic like every service's
API_URL = f"https://documind-api-{NUMBER}.{REGION}.run.app"      # the lane's API, for the last cell's comparison
MEMBER_SA   = f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com"    # on acme, zeta AND globex: the agent NAMES its tenant
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"  # may invoke the service; on no roster
Q = "After how many years of continuous service does gratuity become payable?"   # golden row lk-16: the 1972 Act is in acme's corpus (zeta has the Code's five years; globex has neither)
print("MCP:", MCP_URL)


## Cell 1: The credential, and when it is minted
An ID token lasts an hour. ADK's `McpToolset` takes a `header_provider` - a function it calls on every request - so the token is minted fresh each time instead of being captured once. The audience is the service's root URL, which the server verifies as `SELF_URL`.


In [ ]:
# THE CREDENTIAL, AND WHEN IT IS MINTED. An ID token lasts an hour. Anything that captures one in
# a variable and hands the variable to a long-lived object has a bug with an hour-long fuse: it
# works in the lesson, works in the demo, and starts returning 401 while you are asleep. ADK's
# McpToolset takes a header_provider - a function it calls on EVERY request - so the token is
# minted fresh each time. The audience is the service root, not the /mcp path.
SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, Cell 1).
    A notebook has no metadata server to be anyone with, so it impersonates a roster member; on Cloud
    Run the agent service's own account is the identity and fetch_id_token does the same job."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account, target_scopes=SCOPE)
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def headers_as(service_account: str):
    """A header_provider: called by ADK inside _build_headers on every request (Callable[[ReadonlyContext], dict])."""
    return lambda readonly_context: {"Authorization": f"Bearer {id_token_as(service_account, MCP_URL)}"}

print("token minted, first 24 chars:", id_token_as(MEMBER_SA, MCP_URL)[:24], "...")


## Cell 2: The lane's tools, discovered at runtime


In [ ]:
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

# The lane's tools, discovered at runtime. tool_name_prefix matters the moment there are TWO
# servers: nothing stops two teams publishing a tool called `search`, and the toolset registered
# second silently shadows the first. A prefix makes the collision impossible and the trace readable.
def lane_toolset(service_account: str = MEMBER_SA) -> McpToolset:
    return McpToolset(
        # timeout=120: ADK's default HTTP timeout is 5 s, and a retrieve() through the server is rag-api
        # plus a Gemini answer - up to 90 s cold (7.2). A refusal is instant, which is how a 5 s client
        # passes the roster test and fails the real question. The first live peer did exactly that.
        connection_params=StreamableHTTPConnectionParams(url=f"{MCP_URL}/mcp", timeout=120, sse_read_timeout=300),
        header_provider=headers_as(service_account),
        tool_name_prefix="docs",
    )

lane_tools = lane_toolset()
print("McpToolset created - it connects, and lists the four tools, when the agent first runs")


## Cell 3: The agent
The instruction names the tenant, because `documind-ui-sa` sits on three rosters. That is a *named* tenant the server *checks* - the rule from 7.1 - not a tenant the model chose.


In [ ]:
from google.adk.agents import LlmAgent

# The instruction names the tenant. ui-sa sits on three rosters, so every tenant-scoped tool call carries
# tenant="<name>" - a NAMED tenant the server CHECKS against the roster before the API is called (7.1).
# A per-tenant agent account, on one roster, would need no argument at all.
def instruction_for(tenant: str) -> str:
    return (f"You are DocuMind, answering questions about tenant {tenant}'s documents. "
            f"Pass tenant='{tenant}' to docs_retrieve, docs_list_documents and docs_corpus_stats; docs_calculate_processing_cost takes no tenant. "
            "Use docs_retrieve for any question about the documents and cite the sources it returns; "
            "use docs_list_documents to see what the corpus holds; docs_corpus_stats for counts; "
            "docs_calculate_processing_cost to price a set of pages. "
            "If a tool says the corpus cannot answer, say so and cite nothing. Never state a figure a citation does not carry.")

def agent_for(tenant: str, tools=None, name: str | None = None) -> LlmAgent:
    return LlmAgent(model="gemini-3.6-flash", name=name or f"documind_{tenant}",
                    instruction=instruction_for(tenant), tools=[tools if tools is not None else lane_tools])

agent = agent_for(TENANT, name="documind_agent")
print("agent:", agent.name, "| tenant:", TENANT)


## Cell 4: One turn, one tool call, one cited answer
The assert is the lesson's gate: an agent that answers without calling a tool has answered from memory, which is the failure this module exists to catch.


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

LAST = {"calls": [], "results": [], "text": ""}     # what the last turn did, for the cells that compare

async def run_one_turn(target_agent, question):
    """Run a single agent turn and print every tool call, tool result and text part."""
    LAST.update(calls=[], results=[], text="")
    session_service = InMemorySessionService()
    runner = Runner(agent=target_agent, app_name="documind", session_service=session_service)
    session = await session_service.create_session(app_name="documind", user_id="student")
    content = types.Content(role="user", parts=[types.Part.from_text(text=question)])
    async for event in runner.run_async(user_id="student", session_id=session.id, new_message=content):
        if not (event.content and event.content.parts):
            continue
        for part in event.content.parts:
            if part.function_call:
                LAST["calls"].append(part.function_call.name)
                print(f"  [tool call] {part.function_call.name}({dict(part.function_call.args or {})})")
            if part.function_response:
                LAST["results"].append(part.function_response.response)
                print(f"  [tool result] {str(part.function_response.response)[:160]}")
            if part.text:
                LAST["text"] += part.text
                print(f"  [agent] {part.text.strip()[:500]}")
    return bool(LAST["calls"])

# THIS RUNS. A lesson whose final cell is commented out has not shown you that anything works.
called = await run_one_turn(agent, Q)
assert called, ("The agent answered without calling a tool - the failure this lesson exists to catch. "
                "Is documind-mcp deployed (7.2), is documind-ui-sa allowed to invoke it, is your account "
                "allowed to mint as it (make operators)?")


## Cell 5: Tool chaining


In [ ]:
# Tool chaining: one question, three tools. list_documents tells the agent what the corpus holds,
# corpus_stats how big it is, and the cost tool prices the pages. The agent decides the order; the
# server decides the tenant; the numbers come from the lane, not from the model.
await run_one_turn(agent, "Which documents are indexed for us, how many chunks is that, and what would "
                          "re-processing 119 pages across 5 documents cost at the priority tier?")
print("\ntools this turn:", LAST["calls"])


## Cell 6: One identity, three tenants
The same question, three agents, one account. Acme and Zeta answer with their own handbooks' figures - the golden set's isolation row - and Globex, which holds no handbook, refuses. Isolation is not a setting on the agent; it is the roster and the corpus, checked on every call.


In [ ]:
# The same question for two tenants, from ONE agent identity. ui-sa sits on acme and zeta, so each
# agent names its tenant and the server checks the roster. The golden set's sharpest row (iso-01)
# asks exactly this: the per-trip cap is Rs 40,000 in ACME's handbook and Rs 25,000 in Zeta's.
# Globex holds an MSA and two Acts and no handbook: its corpus cannot answer, and the agent must say so.
CAP = "What is the per-trip cap on domestic travel reimbursement?"
for tenant in ("acme", "zeta", "globex"):
    print(f"\n== {tenant}")
    await run_one_turn(agent_for(tenant), CAP)


## Cell 7: The outsider, from the agent's seat
7.2 showed the refusal from a shell. Here the agent itself is the outsider: the tool call happens, the roster refuses, and the model has to say so. A refused call is still a call - the assert would pass - and that is the point: the agent stays honest because the tool did.


In [ ]:
# The refusal, from the agent's seat. Mint as the eval gate's outsider - allowed to invoke the
# service (7.2 granted it, on purpose), on no roster - and ask the same question. The tool CALL
# happens, so the assert in Cell 4 would still pass: a refused call is still a call. The tool
# RESULT is the roster's refusal, and the model explains it instead of inventing an answer.
outsider_agent = agent_for(TENANT, tools=lane_toolset(OUTSIDER_SA), name="documind_outsider")
called = await run_one_turn(outsider_agent, Q)
print("\ntool called:", called, "| refused by the roster:", any("roster" in str(r) for r in LAST["results"]))


## Cell 8: One corpus, two surfaces
The API called directly, and the agent's tool result: the same chunk ids. ADK hands back the MCP result as a dumped dict, with the server's dict under `structuredContent` and again as JSON text, so the cell digs for `citations` instead of assuming one shape. The server carries the lane's answer; it never makes one.


In [ ]:
# One corpus, two surfaces. The same question straight to the API as documind-ui-sa (4.8's ask()) and
# through the agent a moment ago: the citations name the same chunks. The server carried the lane's
# answer; it never made one. This is the sentence the whole module rests on.
import json

def dig(obj, key):
    """The first value under `key` anywhere in a tool result. ADK hands back the MCP CallToolResult
    dumped as a dict - the server's dict sits under structuredContent, and again as JSON text in
    content[0].text - so the walk parses text that looks like JSON instead of assuming one shape."""
    if isinstance(obj, dict):
        if key in obj: return obj[key]
        for v in obj.values():
            found = dig(v, key)
            if found is not None: return found
    elif isinstance(obj, list):
        for v in obj:
            found = dig(v, key)
            if found is not None: return found
    elif isinstance(obj, str) and obj[:1] in "{[":
        try: return dig(json.loads(obj), key)
        except ValueError: return None
    return None

await run_one_turn(agent, Q)
agent_chunks = {c.get("chunk_id") for r in LAST["results"] for c in (dig(r, "citations") or [])}
r = requests.post(f"{API_URL}/v1/query", json={"query": Q, "tenant_id": TENANT, "top_k": 5, "stream": False},
                  headers={"Authorization": f"Bearer {id_token_as(MEMBER_SA, API_URL)}"}, timeout=90)
api_chunks = {c["chunk_id"] for c in r.json().get("citations", [])}
print("agent cited:", sorted(agent_chunks))
print("API cited  :", sorted(api_chunks))
print("same chunks:", bool(agent_chunks & api_chunks), "- one corpus, two surfaces, one citation")


## Cell 9: The audit line, read back


In [ ]:
# The audit line, read back: every call the server served in the last ten minutes - caller, tenant,
# tool, answerable - the two agents above included. A person's question and an agent's are two rows apart.
since = (datetime.now(timezone.utc) - timedelta(minutes=10)).strftime("%Y-%m-%dT%H:%M:%SZ")
r = subprocess.run(["gcloud", "logging", "read",
                    f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-mcp" AND "mcp_call" AND timestamp>="{since}"',
                    "--project", PROJECT_ID, "--limit", "12",
                    "--format=value(timestamp.date('%H:%M:%S'),jsonPayload.caller,jsonPayload.tenant,jsonPayload.tool,jsonPayload.answerable,textPayload)"],
                   capture_output=True, text=True)
print(r.stdout or "(logs lag a minute - run this cell again)")


## Three identities in one request

```
you (Colab)  --mints as-->  documind-ui-sa  --ID token for documind-mcp-->  documind-mcp
                                                                                |  verifies ui-sa's email, checks the named tenant's roster
                                                                                |  calls retrieve() AS documind-mcp-sa
                                                                                v
                                                                          documind-api  (checks mcp-sa on that tenant's roster)
```

The person who started the notebook, the account the agent speaks as, and the account the server speaks as: three identities, two rosters, one corpus. On the full profile the chat service (12.8) adds a fourth leg - the person's IAP assertion forwarded beside the token - which is how a human's question stays attributed to the human through an agent.

## Other servers, other paths
7.2's second server - the MCP Toolbox over Module 5's warehouse - joins an agent the same way: a second `McpToolset` with its own `header_provider`, a `tool_filter` naming the tools its config defines, and a different prefix. ADK also ships a BigQuery toolset (`google.adk.integrations.bigquery`) that needs no server at all, right for an agent that is the warehouse's only client. 7.1's server on localhost takes the same toolset with the audience set to the local `SELF_URL`. None of them is needed to run this lesson, which is why none of them is here.

## Where Module 8 goes next
8.1 to 8.7's brains call `shared/documind_tools.retrieve` directly - they live inside the kit and need no protocol between them and the lane. The MCP server is for agents *outside* the kit: Claude Desktop, Cursor, another team's runtime, 8.4's A2A peers. Same tools, same roster, same corpus; the protocol is the only thing that changes.


## ✅ Lesson 7.3 complete — and Module 7 with it

- ✅ An ADK agent over the lane's MCP server, with a credential minted per request
- ✅ A named tenant, checked by the server against the roster
- ✅ One turn, one tool call, one cited answer - asserted, not assumed
- ✅ Three tenants from one identity; the outsider refused from the agent's seat
- ✅ The agent and the API cite the same chunk; the audit line shows both

**Next: Module 8 — agents inside the kit, on the one retrieve().**
